# Получение информации по поверке средств измерения в системе "Аршин" через API

Данный гайд показывает как можно получить информацию из системы "Аршин" и проверить информацию по поверке средств измерения используя стандартные библиотеки для работы с данными. Система имеет OEI-API.

OEI-API (Application Programming Interface) - программный интерфейс, предназначенный для предоставления в автоматическом режиме сведений о результатах поверок СИ, содержащихся в Федеральном информационном фонде по обеспечению единства измерений.

API обеспечивает возможность формирования и передачу запроса, и последующее получение результатов запроса в формате JSON. Данная возможность обеспечивается путем предоставления доступа к синхронным интерфейсам с использованием протокола HTTP 1.1.

## Ограничения на этапе тестирования и отладки:

* Выдача ограничена 10000 записей за 1 запрос (LIMIT 10000), планируется отдавать до 3млн. записей
* Вывод данных возможен в виде php array, json или xml. Устанавливается параметром &export_type (1 - php array; 2 - json; 3 - xml)
* Принудительное время выполнения скрипта ограничено 5 минутами. Оптимизируйте запрос под Ваши потребности.
* Планируется возвращение данных в виде сжатого массива
* Регистр при вводе параметр значеняи не имеет
* Все параметры (кроме ?regkey=d37e5f9c2df49556a580b1c3dc8dcc7a), указанные ниже, являются необязательными. Допустимы любые их комбиации
* Параметр regkey=d37e5f9c2df49556a580b1c3dc8dcc7a - ключ доступа для тестирования и отладки. Полноценный рабочий regkey предоставляется бонусом при приобретении годовой подписки на аналитику и будет доступен в профиле пользователя.

Подключим необходимые модули

In [1]:
import pandas as pd
import requests

Зададим точку входа для получения данных

In [2]:
url = 'http://731163-cj72200.tmweb.ru/vri/'

Проверим, что система "Аршин" нам отвечает

In [3]:
r = requests.get(url)
r

<Response [200]>

## Параметры запроса
&export_type - тип выходного массива данных.
1 - php array; 2 - json (для кодирования использована стандартаня PHP функция json_encode(), для декодирования следует использовать json_decode()); 3 - xml. Пример вывода данных в xml формате
http://731163-cj72200.tmweb.ru/vri/?regkey=d37e5f9c2df49556a580b1c3dc8dcc7a&vri_id_from=147222222&vri_id_to=147222322&export_type=3

&vri_id - id поверки.
Число в конце ccылки на карточку поверки в ФГИС АРШИН. Например для поверки https://fgis.gost.ru/fundmetrology/cm/results/1-179725564 это число 179725564

&vri_id_from - id поверки от.
Нижняя граница для поиска поверок по id. Например, при &vri_id_from=12345 будут выводиться только те поверки, чей id больше или равен 12345

&vri_id_to - id поверки до.
Верхняя граница для поиска поверок по id. Например, при &vri_id_to=99999 будут выводиться только те поверки, чей id меньше или равен 99999

&mi_number - Заводской номер СИ.
Текствое поле. Ищется строгое совпадение. Например, для поиска заводсого номера '05359326' необходимо задать &mi_number=05359326

&mi_modification - Модификация СИ.
Текствое поле. Ищется строгое совпадение. Например, для поиска модификации СИ 'Меркурий 230 ART-03 PQRSIDN' необходимо задать &mi_modification=Меркурий 230 ART-03 PQRSIDN

&mitypeTitle - Наименование СИ.
Текствое поле. Можно указывать часть фразы. Например, для поиска наименования СИ 'Счетчики электрической энергии трёхфазные статические' можно задать &mitypeTitle=электрической энергии

&mit_MPISI - Межповерочный интервал в соотвествии с ОТ.
Текствое поле. Можно указывать часть фразы. Например, для поиска такой фразы '4 года - для гор.воды; 6 лет - хол.' можно задать &mit_MPISI=6 лет - хол

&mit_id - id типа СИ в реестре СИ АРШИН.
Поле типа int. Ищется полное совпадение. Например: &mit_id=347162

&mitypeType - Обозначение типа СИ.
Текствое поле. Можно указывать часть фразы. Например, для поиска обозначения типа СИ 'Меркурий 230' можно задать &mitypeType=меркурий

&mitypeNumber - № типа СИ в госреестре.
Текствое поле. Ищется полное совпадение. Например: &mitypeNumber=23345-07

&org_title - Наименование организации-поверителя.
Текствое поле. Можно указывать часть фразы. Например, для поиска организации 'ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ ЭНЕРТЕСТ(ООО ЭНЕРТЕСТ)' можно задать &org_title=энертест

&mi_manufactureYear - Год выпуска СИ.
Целое число. Ищется полное совпадение. Например: &mi_manufactureYear=2009

&mi_signCipher - Условный шифр знака поверки.
Текстовое поле. Ищется полное совпадение. Например: &mi_signCipher=ГЦН

&docTitle - Наиенование методики поверки
Текстовое поле. Можно указывать часть фразы. Например, для документа ГОСТ OIML R 76-1-2011: &docTitle=ГОСТ OIML R 76

&mi_Owner_name - Владелец СИ.
Текствое поле. Можно указывать часть фразы. Например, для поиска организации 'ООО Газпром трансгаз Ухта' можно задать &mi_Owner_name=ООО Газпром

&mit_owner_CountrySI - Страна производства.
Текстовое поле. Ищется полное совпадение. Например: &mit_owner_CountrySI=россия

&mit_owner_SettlementSI - Населенный пункт (производства).
Текствое поле. Можно указывать часть слова или словосочетания. Например, для поиска СИ, произведенных в Москве: &mit_owner_SettlementSI=моск

&mit_owner_ManufacturerSI - Производитель СИ.
Текствое поле. Можно указывать часть фразы. Например, для поиска организации 'ООО Спутник' достаточно задать &mit_owner_ManufacturerSI=Спутник

&poverka_valid_date - Поверка действительна до
Текствое поле. Указывается дата окончания поверки в формате d.m.Y (например - &poverka_valid_date=17.05.2021). Ищется полное совпадение.

&poverka_publication_date - Дата публикации.
Текствое поле. Указывается дата публикации в формате d.m.Y (например - &poverka_publication_date=17.05.2021). Ищется полное совпадение.

&poverka_verification_date - Дата поверки.
Текствое поле. Указывается дата поверки в формате d.m.Y (например - &poverka_verification_date=12.03.2020). Ищется полное совпадение.

&poverka_verification_month - Месяц поверки.
Текствое поле. Указывается месяц поверки с годом в формате m.Y (например - &poverka_verification_month=03.2020). Ищется полное совпадение.

&poverka_verification_year - Год поверки.
Текствое поле. Указывается год в формате Y (например - &poverka_verification_year=2020). Ищется полное совпадение.

&poverka_vriType - Тип поверки.
Числовое поле. 2 - периодическая; 1 - первичная; Например, при такой записи - &poverka_vriType=2 будут отображены только периодические поверки.

## Пример запроса
Давайте запросим все поверенные приборы в МАИ, которые были сделаны в России и поверены в 2021 году.

Запрос займет некоторое время, а также не забывайте указывать регистрационный ключ (ключ в примере получен как тестовый и может не содержать всей информации из реестра)

In [4]:
keys = {'regkey': 'd37e5f9c2df49556a580b1c3dc8dcc7a',
        'mi_Owner_name': 'Новосибирский авиационный завод',
        #'poverka_verification_year': '2024',
        'export_type': '2'}

r = requests.get(url, params=keys)
r

<Response [200]>

Прочитаем данные из запроса

In [5]:
try:
    json = r.json()
except ValueError:
    print("Oops!")

Переведем данные в удобный нам формат Pandas DataFrame

In [6]:
df = pd.DataFrame(json)
df

,id,vri_id,mi_number,mi_modification,mi_manufactureYear,mi_signCipher,mi_Owner_name,org_title,fsa_ral_regNumbers_regNumber,mitypeNumber,...,vriType,result_docnum,result_doc_type,additional_info,means_npe,means_uve,means_mieta,means_ses,means_mis,means_reagent
0,1,41942091,7727,Е6-24/1,NaN,Н,Филиа...,Запад...,RA.RU...,25405-08,...,Периодическая,C-Н/0...,Извещение о непригодности,,,,56598.14.3Р.00198402 56598-14 Магазины сопроти...,,2303-68 Киловольтметры электростатические (№53...,
1,2,43465375,5299,КИСС-03,2018,Н,Филиа...,Запад...,RA.RU...,20641-11,...,Периодическая,C-Н/0...,Извещение о непригодности,,,,54727.13.2Р.00117862 54727-13 Компараторы-кали...,,1162-58 Катушки электрического сопротивления и...,
2,3,42536520,12480,М244,1970,Н,Филиа...,Запад...,RA.RU...,2373-68,...,Периодическая,C-Н/0...,Извещение о непригодности,,,,55804.13.1Р.00108519 55804-13 Калибраторы мног...,,,
3,4,38122750,1401,нет м...,2014,Н,Новос...,Запад...,RA.RU...,47965-11,...,Периодическая,C-Н/1...,Извещение о непригодности,,,3.1.ZZН.0046.2012 ГЭЕ длины 1 разряда в диапаз...,,,,
4,5,45933349,651358,КО-1,NaN,Н,Новос...,Запад...,RA.RU...,868-72,...,Периодическая,C-Н/1...,Извещение о непригодности,,,3.1.ZZН.0050.2013 ГЭЕ плоского угла 2 разряда ...,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,97,354894349,24211,Сейтр...,NaN,ВМ,ПЕРВИ...,ФЕДЕР...,RA.RU...,27033-13,...,Периодическая,C-ВМ/...,Извещение о непригодности,,,,46835.11.1Р.99495 46835-11 Меры профильные; 46...,,27015-04 Комплекты поверки гирь и весов перено...,
97,98,354964012,24210,Сейтр...,NaN,ВМ,ПЕРВИ...,ФЕДЕР...,RA.RU...,27033-13,...,Периодическая,C-ВМ/...,Извещение о непригодности,,,,46835.11.1Р.99495 46835-11 Меры профильные; 46...,,27015-04 Комплекты поверки гирь и весов перено...,
98,99,355973023,24210,Сейтр...,NaN,ВМ,ПЕРВИ...,ФЕДЕР...,RA.RU...,27033-13,...,Первичная поверка,C-ВМ/...,Извещение о непригодности,,,,46835.11.1Р.99495 46835-11 Меры профильные; 46...,,27015-04 Комплекты поверки гирь и весов перено...,
99,100,355973024,24211,Сейтр...,NaN,ВМ,ПЕРВИ...,ФЕДЕР...,RA.RU...,27033-13,...,Первичная поверка,C-ВМ/...,Извещение о непригодности,,,,46835.11.1Р.99495 46835-11 Меры профильные; 46...,,27015-04 Комплекты поверки гирь и весов перено...,


In [7]:
df['mitypeURL'][66]

'https://fgis.gost.ru/fundmetrology/registry/4/items/345372'

# Практическое задания
* Попробуйте получить данные о манометрах, поверенных в ЦАГИ имени Н.Е. Жуковского
* Проверьте есть ли в МАИ поверенные расходомеры
* Проверьте свой домашний счетчик воды (горячей или холодной) на наличие поверки, если конечно счетчик у вас установлен &#x1F600; и он был поверен после 24.09.2020 года (именно с этой даты все организации обязаны передавать данные о поверке в единую систему).

Все данные по п.1/2 обработайте и сведите в Pandas Dataframe в формат удобный для датасета по оценке запросов на поверку средств измерения

Постройте аналитику по полученным данным.

In [8]:
import pandas as pd
import requests
from IPython.display import display

url = 'http://731163-cj72200.tmweb.ru/vri/'
REGKEY = 'd37e5f9c2df49556a580b1c3dc8dcc7a'


def get_data_from_arshin(params: dict, url: str = url) -> pd.DataFrame:
    """
    Получает данные из API Аршин и возвращает Pandas DataFrame.
    """
    params = params.copy()
    params.setdefault('regkey', REGKEY)
    params.setdefault('export_type', '2')

    response = requests.get(url, params=params, timeout=120)

    print('URL запроса:')
    print(response.url)
    print('Код ответа:', response.status_code)

    try:
        data = response.json()
    except ValueError:
        print('Ошибка: ответ не является JSON')
        print(response.text[:1000])
        return pd.DataFrame()

    if isinstance(data, dict):
        for value in data.values():
            if isinstance(value, list):
                data = value
                break
        else:
            return pd.DataFrame([data])

    if not data:
        return pd.DataFrame()

    return pd.DataFrame(data)

In [9]:
csagi_manometers_keys = {
    'mitypeTitle': 'манометр',
    'org_title': 'ЦАГИ',
    'export_type': '2'
}

df_csagi_manometers = get_data_from_arshin(csagi_manometers_keys)

print('Количество найденных записей:', len(df_csagi_manometers))
display(df_csagi_manometers.head())

URL запроса:
http://731163-cj72200.tmweb.ru/vri/?mitypeTitle=%D0%BC%D0%B0%D0%BD%D0%BE%D0%BC%D0%B5%D1%82%D1%80&org_title=%D0%A6%D0%90%D0%93%D0%98&export_type=2&regkey=d37e5f9c2df49556a580b1c3dc8dcc7a
Код ответа: 200
Количество найденных записей: 5837


,id,vri_id,mi_number,mi_modification,mi_manufactureYear,mi_signCipher,mi_Owner_name,org_title,fsa_ral_regNumbers_regNumber,mitypeNumber,...,vriType,result_docnum,result_doc_type,additional_info,means_npe,means_uve,means_mieta,means_ses,means_mis,means_reagent
0,1,1173224226,01-К,,NaN,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
1,2,1173224227,02-К,,NaN,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
2,3,1173224228,03-К,,NaN,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
3,4,1173224229,04-К,,NaN,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,
4,5,1173224230,05-К,,NaN,АОЛ,,ФГУП ...,РОСС ...,55372-13,...,Без статуса,нет д...,Извещение о непригодности,,,3.1.АОЛ.0080.2016 Государственный эталон едини...,,,,


In [10]:
df_csagi_manometers.columns

Index(['id', 'vri_id', 'mi_number', 'mi_modification', 'mi_manufactureYear',
       'mi_signCipher', 'mi_Owner_name', 'org_title',
       'fsa_ral_regNumbers_regNumber', 'mitypeNumber', 'mitypeTitle',
       'mit_owner_CountrySI', 'mit_owner_SettlementSI',
       'mit_owner_ManufacturerSI', 'mit_MP_doc', 'mit_MP_link', 'mit_MPISI',
       'docTitle', 'mit_id', 'mitypeURL', 'mitypeType', 'verification_date',
       'valid_date', 'publication_date', 'vriType', 'result_docnum',
       'result_doc_type', 'additional_info', 'means_npe', 'means_uve',
       'means_mieta', 'means_ses', 'means_mis', 'means_reagent'],
      dtype='str')

In [11]:
important_columns = [
    'vri_id',
    'mi_number',
    'mi_modification',
    'mi_manufactureYear',
    'mi_Owner_name',
    'org_title',
    'mitypeTitle',
    'mitypeType',
    'mitypeNumber',
    'poverka_verification_date',
    'poverka_valid_date',
    'vriType',
    'result_doc_type',
    'result_docnum',
    'mitypeURL'
]

existing_columns = [col for col in important_columns if col in df_csagi_manometers.columns]

df_csagi_manometers_short = df_csagi_manometers[existing_columns].copy()
df_csagi_manometers_short.insert(0, 'task', 'Манометры, поверенные в ЦАГИ')

display(df_csagi_manometers_short.head())

,task,vri_id,mi_number,mi_modification,mi_manufactureYear,mi_Owner_name,org_title,mitypeTitle,mitypeType,mitypeNumber,vriType,result_doc_type,result_docnum,mitypeURL
0,"Манометры, поверенные в ЦАГИ",1173224226,01-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...
1,"Манометры, поверенные в ЦАГИ",1173224227,02-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...
2,"Манометры, поверенные в ЦАГИ",1173224228,03-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...
3,"Манометры, поверенные в ЦАГИ",1173224229,04-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...
4,"Манометры, поверенные в ЦАГИ",1173224230,05-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...


ЗАДАЧА 2


In [12]:
mai_flowmeter_queries = [
    {
        'mitypeTitle': 'расходомер',
        'mi_Owner_name': 'МАИ',
        'export_type': '2'
    },
    {
        'mitypeTitle': 'расходомер',
        'mi_Owner_name': 'Московский авиационный институт',
        'export_type': '2'
    },
    {
        'mitypeTitle': 'расходомер',
        'org_title': 'МАИ',
        'export_type': '2'
    },
    {
        'mitypeTitle': 'расходомер',
        'org_title': 'Московский авиационный институт',
        'export_type': '2'
    }
]

mai_flowmeter_frames = []

for i, query in enumerate(mai_flowmeter_queries, start=1):
    print(f'\n--- Запрос {i} ---')
    df_temp = get_data_from_arshin(query)

    if not df_temp.empty:
        df_temp = df_temp.copy()
        df_temp['search_variant'] = f'Запрос {i}'
        mai_flowmeter_frames.append(df_temp)

if mai_flowmeter_frames:
    df_mai_flowmeters = pd.concat(mai_flowmeter_frames, ignore_index=True)

    if 'vri_id' in df_mai_flowmeters.columns:
        df_mai_flowmeters = df_mai_flowmeters.drop_duplicates(subset='vri_id')

else:
    df_mai_flowmeters = pd.DataFrame()

print('Количество найденных расходомеров в МАИ:', len(df_mai_flowmeters))
display(df_mai_flowmeters.head())


--- Запрос 1 ---
URL запроса:
http://731163-cj72200.tmweb.ru/vri/?mitypeTitle=%D1%80%D0%B0%D1%81%D1%85%D0%BE%D0%B4%D0%BE%D0%BC%D0%B5%D1%80&mi_Owner_name=%D0%9C%D0%90%D0%98&export_type=2&regkey=d37e5f9c2df49556a580b1c3dc8dcc7a
Код ответа: 200

--- Запрос 2 ---
URL запроса:
http://731163-cj72200.tmweb.ru/vri/?mitypeTitle=%D1%80%D0%B0%D1%81%D1%85%D0%BE%D0%B4%D0%BE%D0%BC%D0%B5%D1%80&mi_Owner_name=%D0%9C%D0%BE%D1%81%D0%BA%D0%BE%D0%B2%D1%81%D0%BA%D0%B8%D0%B9+%D0%B0%D0%B2%D0%B8%D0%B0%D1%86%D0%B8%D0%BE%D0%BD%D0%BD%D1%8B%D0%B9+%D0%B8%D0%BD%D1%81%D1%82%D0%B8%D1%82%D1%83%D1%82&export_type=2&regkey=d37e5f9c2df49556a580b1c3dc8dcc7a
Код ответа: 200

--- Запрос 3 ---
URL запроса:
http://731163-cj72200.tmweb.ru/vri/?mitypeTitle=%D1%80%D0%B0%D1%81%D1%85%D0%BE%D0%B4%D0%BE%D0%BC%D0%B5%D1%80&org_title=%D0%9C%D0%90%D0%98&export_type=2&regkey=d37e5f9c2df49556a580b1c3dc8dcc7a
Код ответа: 200

--- Запрос 4 ---
URL запроса:
http://731163-cj72200.tmweb.ru/vri/?mitypeTitle=%D1%80%D0%B0%D1%81%D1%85%D0%BE%D0%B4%D

,id,vri_id,mi_number,mi_modification,mi_manufactureYear,mi_signCipher,mi_Owner_name,org_title,fsa_ral_regNumbers_regNumber,mitypeNumber,...,result_docnum,result_doc_type,additional_info,means_npe,means_uve,means_mieta,means_ses,means_mis,means_reagent,search_variant
0,1,427990506,M1520...,EL-FLOW,2015,ДШЛ,НИИ П...,ОБЩЕС...,RA.RU...,25705-10,...,C-ДШЛ...,Извещение о непригодности,,,,40432.09.1Р.00334741 40432-09 Стенд для калибр...,,,,Запрос 1
1,2,427990697,M1520...,EL-FLOW,2015,ДШЛ,НИИ П...,ОБЩЕС...,RA.RU...,25705-10,...,C-ДШЛ...,Извещение о непригодности,,,,40432.09.1Р.00334741 40432-09 Стенд для калибр...,,,,Запрос 1
2,3,427990440,M1721...,EL-FLOW,2017,ДШЛ,НИИ П...,ОБЩЕС...,RA.RU...,64700-16,...,C-ДШЛ...,Извещение о непригодности,,,,40432.09.1Р.00334741 40432-09 Стенд для калибр...,,,,Запрос 1
3,4,427990539,M1721...,EL-FLOW,2017,ДШЛ,НИИ П...,ОБЩЕС...,RA.RU...,64700-16,...,C-ДШЛ...,Извещение о непригодности,,,,40432.09.1Р.00334741 40432-09 Стенд для калибр...,,,,Запрос 1
4,5,172632977,161954,Питер...,NaN,БЯ,МАИ+3Н,ФЕДЕР...,RA.RU...,46814-11,...,C-БЯ/...,Извещение о непригодности,"0,25 л/имп",,,53155.13.2Р.00163517 53155-13 Установки пролив...,,,,Запрос 1


In [13]:
existing_columns = [col for col in important_columns if col in df_mai_flowmeters.columns]

if not df_mai_flowmeters.empty:
    df_mai_flowmeters_short = df_mai_flowmeters[existing_columns].copy()
    df_mai_flowmeters_short.insert(0, 'task', 'Расходомеры в МАИ')
else:
    df_mai_flowmeters_short = pd.DataFrame(columns=['task'] + existing_columns)

display(df_mai_flowmeters_short.head())

,task,vri_id,mi_number,mi_modification,mi_manufactureYear,mi_Owner_name,org_title,mitypeTitle,mitypeType,mitypeNumber,vriType,result_doc_type,result_docnum,mitypeURL
0,Расходомеры в МАИ,427990506,M1520...,EL-FLOW,2015,НИИ П...,ОБЩЕС...,Расхо...,EL-Fl...,25705-10,Периодическая,Извещение о непригодности,C-ДШЛ...,https://fgis.gost.ru/fundmetrology/registry/4/...
1,Расходомеры в МАИ,427990697,M1520...,EL-FLOW,2015,НИИ П...,ОБЩЕС...,Расхо...,EL-Fl...,25705-10,Периодическая,Извещение о непригодности,C-ДШЛ...,https://fgis.gost.ru/fundmetrology/registry/4/...
2,Расходомеры в МАИ,427990440,M1721...,EL-FLOW,2017,НИИ П...,ОБЩЕС...,Расхо...,EL-FL...,64700-16,Периодическая,Извещение о непригодности,C-ДШЛ...,https://fgis.gost.ru/fundmetrology/registry/4/...
3,Расходомеры в МАИ,427990539,M1721...,EL-FLOW,2017,НИИ П...,ОБЩЕС...,Расхо...,EL-FL...,64700-16,Периодическая,Извещение о непригодности,C-ДШЛ...,https://fgis.gost.ru/fundmetrology/registry/4/...
4,Расходомеры в МАИ,172632977,161954,Питер...,NaN,МАИ+3Н,ФЕДЕР...,Расхо...,Питер...,46814-11,Периодическая,Извещение о непригодности,C-БЯ/...,https://fgis.gost.ru/fundmetrology/registry/4/...


In [14]:
if df_mai_flowmeters.empty:
    print('По заданным параметрам поверенные расходомеры в МАИ не найдены.')
else:
    print(f'Поверенные расходомеры в МАИ найдены. Количество записей: {len(df_mai_flowmeters)}')

Поверенные расходомеры в МАИ найдены. Количество записей: 25


Общий датафрейм по задачам 1 и 2


In [15]:
df_tasks_1_2 = pd.concat(
    [
        df_csagi_manometers_short,
        df_mai_flowmeters_short
    ],
    ignore_index=True
)

print('Размер итогового датасета:', df_tasks_1_2.shape)
display(df_tasks_1_2.head(20))

Размер итогового датасета: (5862, 14)


,task,vri_id,mi_number,mi_modification,mi_manufactureYear,mi_Owner_name,org_title,mitypeTitle,mitypeType,mitypeNumber,vriType,result_doc_type,result_docnum,mitypeURL
0,"Манометры, поверенные в ЦАГИ",1173224226,01-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...
1,"Манометры, поверенные в ЦАГИ",1173224227,02-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...
2,"Манометры, поверенные в ЦАГИ",1173224228,03-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...
3,"Манометры, поверенные в ЦАГИ",1173224229,04-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...
4,"Манометры, поверенные в ЦАГИ",1173224230,05-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...
5,"Манометры, поверенные в ЦАГИ",1173224231,06-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...
6,"Манометры, поверенные в ЦАГИ",1173224232,07-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...
7,"Манометры, поверенные в ЦАГИ",1173224233,08-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...
8,"Манометры, поверенные в ЦАГИ",1173224234,09-К,,NaN,,ФГУП ...,Маном...,P1454,55372-13,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...
9,"Манометры, поверенные в ЦАГИ",1173224235,220396,,NaN,,ФГУП ...,Маном...,ОБМ1-...,1784-63,Без статуса,Извещение о непригодности,нет д...,https://fgis.gost.ru/fundmetrology/registry/4/...


In [17]:
df_tasks_1_2.to_csv('arshin_tasks_1_2_dataset.csv', index=False, encoding='utf-8-sig')
df_tasks_1_2.to_excel('arshin_tasks_1_2_dataset.xlsx', index=False)

print('Файлы сохранены:')
print('arshin_tasks_1_2_dataset.csv')
print('arshin_tasks_1_2_dataset.xlsx')

Файлы сохранены:
arshin_tasks_1_2_dataset.csv
arshin_tasks_1_2_dataset.xlsx


Аналитика по полученным данным

In [18]:
print('Общее количество записей:', len(df_tasks_1_2))

print('\nКоличество записей по задачам:')
display(df_tasks_1_2['task'].value_counts())

Общее количество записей: 5862

Количество записей по задачам:


task
Манометры, поверенные в ЦАГИ    5837
Расходомеры в МАИ                 25
Name: count, dtype: int64

In [19]:
if 'vriType' in df_tasks_1_2.columns:
    print('Типы поверки:')
    display(df_tasks_1_2['vriType'].value_counts(dropna=False))

Типы поверки:


vriType
Периодическая        3063
Без статуса          2798
Первичная поверка       1
Name: count, dtype: int64

In [20]:
if 'result_doc_type' in df_tasks_1_2.columns:
    print('Типы документов по результатам поверки:')
    display(df_tasks_1_2['result_doc_type'].value_counts(dropna=False))

Типы документов по результатам поверки:


result_doc_type
Извещение о непригодности    5862
Name: count, dtype: int64

In [21]:
if 'org_title' in df_tasks_1_2.columns:
    print('Организации-поверители:')
    display(df_tasks_1_2['org_title'].value_counts(dropna=False).head(10))

Организации-поверители:


org_title
ФЕДЕР...    3615
ФГУП ...    2235
ОБЩЕС...      10
Федер...       2
Name: count, dtype: int64

In [22]:
if 'mi_Owner_name' in df_tasks_1_2.columns:
    print('Владельцы средств измерений:')
    display(df_tasks_1_2['mi_Owner_name'].value_counts(dropna=False).head(10))

Владельцы средств измерений:


mi_Owner_name
            2508
ФГУП ...    1381
-           1310
ФАУ &...     496
Специ...     115
юриди...      27
МАИ+3Н         8
НИИ П...       4
ИП Ба...       2
Индив...       2
Name: count, dtype: int64

In [23]:
if 'mi_manufactureYear' in df_tasks_1_2.columns:
    print('Распределение по годам выпуска средств измерений:')
    display(
        df_tasks_1_2['mi_manufactureYear']
        .value_counts(dropna=False)
        .sort_index()
    )

Распределение по годам выпуска средств измерений:


mi_manufactureYear
1972       1
1977       1
1978       4
1980       8
1985       2
2015       4
2017       4
NaN     5838
Name: count, dtype: int64

In [24]:
if 'mitypeTitle' in df_tasks_1_2.columns:
    print('Наименования типов средств измерений:')
    display(df_tasks_1_2['mitypeTitle'].value_counts(dropna=False).head(10))

Наименования типов средств измерений:


mitypeTitle
Маном...    5830
Расхо...      24
Микро...       5
Тягом...       2
Счетч...       1
Name: count, dtype: int64

Выводы

In [25]:
total_records = len(df_tasks_1_2)
csagi_count = len(df_csagi_manometers_short)
mai_count = len(df_mai_flowmeters_short)

print('Вывод:')
print(f'В ходе работы были получены данные из API системы "Аршин".')
print(f'По запросу манометров, поверенных в ЦАГИ, найдено записей: {csagi_count}.')
print(f'По запросу расходомеров в МАИ найдено записей: {mai_count}.')
print(f'Итоговый датасет по первым двум заданиям содержит {total_records} записей.')

if mai_count == 0:
    print('По заданным параметрам расходомеры в МАИ не обнаружены. Возможная причина — отсутствие записей в базе или другое написание организации в реестре.')
else:
    print('Наличие записей по расходомерам в МАИ подтверждает, что такие средства измерений присутствуют в выгрузке.')

print('Полученные данные были приведены к табличному виду Pandas DataFrame и могут использоваться для дальнейшего анализа поверок средств измерений.')

Вывод:
В ходе работы были получены данные из API системы "Аршин".
По запросу манометров, поверенных в ЦАГИ, найдено записей: 5837.
По запросу расходомеров в МАИ найдено записей: 25.
Итоговый датасет по первым двум заданиям содержит 5862 записей.
Наличие записей по расходомерам в МАИ подтверждает, что такие средства измерений присутствуют в выгрузке.
Полученные данные были приведены к табличному виду Pandas DataFrame и могут использоваться для дальнейшего анализа поверок средств измерений.


ЗАДАЧА 3


In [64]:
def load_arshin_df(base_url: str, params: dict) -> pd.DataFrame:
    payload = {'regkey': REGKEY, 'export_type': '2'}
    payload.update(params)

    response = requests.get(base_url, params=payload, timeout=120)
    response.raise_for_status()

    data = response.json()
    if isinstance(data, dict):
        # На случай, если API вернет объект с вложенным списком
        # (например, под ключом data/items/results)
        for key in ('data', 'items', 'results'):
            if key in data and isinstance(data[key], list):
                data = data[key]
                break

    return pd.DataFrame(data)

my_water_meter_params = {
    'mitypeTitle': 'Счетчик',
    'mi_number': '23131979',
    #'mi_Owner_name': 'ФИО',
    #'poverka_verification_year': '2026',
}

if my_water_meter_params['mi_number'] != 'тестик':
    my_meter_df = load_arshin_df(url, my_water_meter_params)

    if my_meter_df.empty:
        print('По вашему счетчику записи о поверке не найдены.')
    else:
        print(f"Найдено записей по вашему счетчику: {len(my_meter_df)}")
        display(my_meter_df.head())
else:
    print('Заполните номер')

Найдено записей по вашему счетчику: 3


,id,vri_id,mi_number,mi_modification,mi_manufactureYear,mi_signCipher,mi_Owner_name,org_title,fsa_ral_regNumbers_regNumber,mitypeNumber,...,vriType,result_docnum,result_doc_type,additional_info,means_npe,means_uve,means_mieta,means_ses,means_mis,means_reagent
0,1,140990596,23131979,СГВ-15,NaN,ДКА,Степа...,ОБЩЕС...,RA.RU...,16078-13,...,Периодическая,C-ДКА...,Извещение о непригодности,,,,40391.09.3Р.00320899 40391-09 Установки поверо...,,,
1,2,255923319,23131979,НАРТИ...,2023,МА,ООО «...,ФЕДЕР...,RA.RU...,77904-20,...,Первичная поверка,C-МА/...,Извещение о непригодности,-,,,49992.12.2Р.00536775 49992-12 Установки автома...,,,
2,3,261813535,23131979,СТК М...,NaN,ДЗЫ,-,ОБЩЕС...,RA.RU...,75191-19,...,Первичная поверка,C-ДЗЫ...,Извещение о непригодности,,,3.2.ГКХ.0014.2017 Эталон единицы температуры в...,60684.15.1Р.00819932 60684-15 Установки поверо...,,,
